# Task 3: Controlled Vietnamese Pedagogical Feedback — QLoRA Training & Evaluation Pipeline

Pipeline huấn luyện và đánh giá mô hình **Qwen2.5-Coder-7B-Instruct** cho bài toán **Sinh phản hồi sư phạm tiếng Việt có kiểm soát mức độ (Task 3)**:
- **Chiến lược dữ liệu**: Huấn luyện kết hợp (Mixed Balanced SFT) trên dữ liệu thực (`real-train`) và dữ liệu tổng hợp đã qua thẩm định (`synthetic-train`).
- **Kiểm soát an toàn (Compliance Guard)**: Giám sát vi phạm rò rỉ mã nguồn/lời giải ở Level 1 và Level 2 nhằm đảm bảo tính sư phạm và tối ưu điểm số đánh giá.
- **Đóng gói kết quả**: Suy luận trên tập kiểm thử công khai (`public_test_release`), tự động trích xuất và xác thực định dạng nộp bài chuẩn theo yêu cầu của hệ thống chấm điểm.

---
### Hướng dẫn thiết lập môi trường:
1. **Accelerator**: Chọn **GPU T4 x2** hoặc **GPU P100** (bắt buộc cho tiến trình huấn luyện QLoRA 4-bit).
2. **Internet**: Bật **On** để tải mô hình nền và thư viện phụ thuộc.
3. **Dữ liệu đầu vào**:
   - Tải lên các gói dữ liệu (`generate-dataset.zip`, `sample_dataset.zip`, `public_test_release.zip`) thông qua tính năng **+ Add Data**, hoặc chỉ định mã định danh Google Drive tương ứng trong ô cấu hình.


In [ ]:
# ==============================================================================
# 1. THIẾT LẬP THAM SỐ THỰC NGHIỆM (PIPELINE CONFIGURATION)
# ==============================================================================

# Cấu hình mã nguồn & nhánh làm việc
REPO = "github.com/tiendinhquang2104/llm-final-project.git"
BRANCH = "task-3"

# Định danh tệp lưu trữ Google Drive (sử dụng khi không tải trực tiếp qua Kaggle Input)
DRIVE_ID_PUBLIC_SAMPLE = "1gfMQ9LpCbg2AOwPfHnt7qe220J1AfLBj"  # Gói sample_dataset & public_test_release
DRIVE_ID_SYNTHETIC = ""                                      # Gói generate-dataset (nếu có trên Drive)

# Chế độ thực thi:
# - "qwen_qlora": Huấn luyện QLoRA trên tập dữ liệu kết hợp và sinh kết quả nộp bài
# - "qwen_baseline": Đánh giá suy luận trực tiếp với mô hình nền (Zero-shot / In-context)
# - "gemini": Đánh giá baseline thông qua Google Gemini API
MODE = "qwen_qlora"

# Tập dữ liệu mục tiêu cho bước đánh giá cuối:
# - "public": Chạy suy luận trên toàn bộ 328 bài nộp của Public Test và đóng gói submission
# - "real-val": Đánh giá thẩm định nội bộ trên tập kiểm thử chuẩn 6 mẫu có nhãn chuyên gia
EVAL_SPLIT = "public"

# Tham số tối ưu hóa mô hình:
EXPERIMENT = "mixed_balanced"  # Chiến lược cân bằng phân phối giữa real-train và synthetic-train
EPOCHS = 3                     # Số chu kỳ huấn luyện
MAX_STEPS = None               # Giới hạn số bước cập nhật tham số (None: huấn luyện toàn diện)

WORK = "/kaggle/tmp"


In [ ]:
# ==============================================================================
# 2. KHỞI TẠO MÃ NGUỒN VÀ THIẾT LẬP BIẾN MÔI TRƯỜNG
# ==============================================================================

import os, sys, subprocess
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
def get_secret(name):
    try:
        return secrets.get_secret(name)
    except Exception:
        return ""

token = get_secret("GITHUB_TOKEN")
url = f"https://{token}@{REPO}" if token else f"https://{REPO}"

os.makedirs(WORK, exist_ok=True)
REPO_DIR = f"{WORK}/repo"

if not os.path.exists(REPO_DIR):
    print(f"Khởi tạo kho lưu trữ {REPO} (nhánh: {BRANCH})...")
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("Không thể đồng bộ mã nguồn — vui lòng kiểm tra nhánh hoặc quyền truy cập.") from None
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", f"https://{REPO}"], check=True)
else:
    print(f"Đồng bộ bản cập nhật mới nhất từ nhánh {BRANCH}...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q", "origin", BRANCH])

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Đồng bộ thông tin xác thực từ Kaggle Secrets vào tệp cấu hình cục bộ .env
gemini_key = get_secret("GEMINI_API_KEY")
openai_key = get_secret("OPENAI_API_KEY")
hf_token = get_secret("HF_TOKEN")

env_lines = []
if gemini_key:
    os.environ["GEMINI_API_KEY"] = gemini_key
    env_lines.append(f"GEMINI_API_KEY={gemini_key}")
if openai_key:
    os.environ["OPENAI_API_KEY"] = openai_key
    env_lines.append(f"OPENAI_API_KEY={openai_key}")
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    env_lines.append(f"HF_TOKEN={hf_token}")

if env_lines:
    with open(".env", "w", encoding="utf-8") as f:
        f.write("\n".join(env_lines) + "\n")

print("Môi trường thực thi:", os.getcwd())
print("Bản ghi commit hiện tại:", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())


In [ ]:
# ==============================================================================
# 3. CÀI ĐẶT THƯ VIỆN PHỤ THUỘC VÀ XÁC MINH PHẦN CỨNG TĂNG TỐC
# ==============================================================================

!pip install -q gdown pyyaml pytest
!pip install -q "transformers>=4.51,<5" accelerate bitsandbytes peft trl datasets
if MODE in ("gemini", "openai"):
    !pip install -q google-genai openai

# Kiểm tra thông số trạng thái thiết bị GPU
!nvidia-smi


In [ ]:
# ==============================================================================
# 4. TIỀN XỬ LÝ VÀ CHUẨN HÓA DỮ LIỆU ĐA NGUỒN (DATA PIPELINE PREPARATION)
# ==============================================================================

import os, glob, shutil, gdown

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/synth", exist_ok=True)
os.makedirs(f"{WORK}/download", exist_ok=True)

# ------------------------------------------------------------------------------
# 4.1. Chuẩn bị tập dữ liệu thực nghiệm và tập kiểm thử công khai
# ------------------------------------------------------------------------------
print("=== [1/3] Trích xuất dữ liệu thực và tập kiểm thử công khai ===")
found_sample = glob.glob("/kaggle/input/**/sample_dataset", recursive=True)
found_public = glob.glob("/kaggle/input/**/public_test_release", recursive=True)
found_real_zips = glob.glob("/kaggle/input/**/sample_dataset*.zip", recursive=True) or glob.glob("/kaggle/input/**/*cham-diem*.zip", recursive=True)

if found_sample and not os.path.exists("data/raw/sample_dataset"):
    shutil.copytree(found_sample[0], "data/raw/sample_dataset")
    print(f"-> Nạp sample_dataset từ: {found_sample[0]}")

if found_public and not os.path.exists("data/raw/public_test_release"):
    shutil.copytree(found_public[0], "data/raw/public_test_release")
    print(f"-> Nạp public_test_release từ: {found_public[0]}")

if not os.path.exists("data/raw/sample_dataset") or not os.path.exists("data/raw/public_test_release"):
    zip_source = None
    if found_real_zips:
        zip_source = found_real_zips[0]
        print(f"-> Sử dụng lưu trữ nén nội bộ: {zip_source}")
    else:
        drive_id = DRIVE_ID_PUBLIC_SAMPLE or get_secret("PUBLIC_TEST_DRIVE_ID") or get_secret("DRIVE_ID")
        if drive_id:
            zip_dest = f"{WORK}/download/public_and_sample.zip"
            print(f"-> Tải tệp lưu trữ từ máy chủ đám mây (ID: {drive_id})...")
            gdown.download(id=drive_id, output=zip_dest, quiet=False)
            zip_source = zip_dest

    if zip_source and os.path.exists(zip_source):
        !python scripts/unpack_data.py {os.path.dirname(zip_source)} --dest data/raw
        for sub_zip in glob.glob("data/raw/*.zip"):
            !unzip -q -o {sub_zip} -d data/raw

!rm -rf data/raw/__MACOSX

# ------------------------------------------------------------------------------
# 4.2. Chuẩn bị tập dữ liệu tổng hợp (Synthetic Dataset)
# ------------------------------------------------------------------------------
print("\n=== [2/3] Trích xuất dữ liệu tổng hợp kiểm định luật ===")
synth_dest = "data/synth/generate-dataset"
found_synth_dirs = [d for d in glob.glob("/kaggle/input/**/generate-dataset*", recursive=True) if os.path.isdir(d) and os.path.exists(os.path.join(d, "task3_feedback.json"))]
found_synth_zips = glob.glob("/kaggle/input/**/generate-dataset*.zip", recursive=True)

if found_synth_dirs and not os.path.exists(synth_dest):
    shutil.copytree(found_synth_dirs[0], synth_dest)
    print(f"-> Nạp generate-dataset từ: {found_synth_dirs[0]}")
elif found_synth_zips and not os.path.exists(synth_dest):
    print(f"-> Giải nén generate-dataset từ lưu trữ: {found_synth_zips[0]}")
    !unzip -q -o {found_synth_zips[0]} -d data/synth
else:
    drive_synth_id = DRIVE_ID_SYNTHETIC or get_secret("SYNTHETIC_DRIVE_ID")
    if drive_synth_id:
        synth_zip = f"{WORK}/download/generate_dataset.zip"
        print(f"-> Tải gói dữ liệu tổng hợp từ máy chủ đám mây (ID: {drive_synth_id})...")
        gdown.download(id=drive_synth_id, output=synth_zip, quiet=False)
        !unzip -q -o {synth_zip} -d data/synth
    elif not os.path.exists(synth_dest):
        print("Thông báo: Chưa phát hiện thư mục generate-dataset. Có thể bổ sung qua tính năng '+ Add Data'.")

# Chuẩn hoá cây thư mục nếu tồn tại phân cấp lồng nhau
if os.path.exists(f"{synth_dest}/generate-dataset"):
    for item in os.listdir(f"{synth_dest}/generate-dataset"):
        shutil.move(f"{synth_dest}/generate-dataset/{item}", f"{synth_dest}/{item}")
    shutil.rmtree(f"{synth_dest}/generate-dataset")

!rm -rf data/synth/__MACOSX

# ------------------------------------------------------------------------------
# 4.3. Phân vùng và thẩm định tính toàn vẹn dữ liệu
# ------------------------------------------------------------------------------
print("\n=== [3/3] Chuẩn hóa và xác thực phân vùng dữ liệu Task 3 ===")
!python scripts/task3_prepare.py --filter-synthetic strict

print("\nThống kê các phân vùng dữ liệu hợp lệ (data/processed/task3):")
!ls -lh data/processed/task3


In [ ]:
# ==============================================================================
# 5. CẤU HÌNH THIẾT BỊ TÍNH TOÁN VÀ QUẢN TRỊ BỘ NHỚ CUDA
# ==============================================================================

import os, torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gpu_count = torch.cuda.device_count()
print(f"Hệ thống phát hiện {gpu_count} GPU khả dụng:")
for i in range(gpu_count):
    free_mem = torch.cuda.mem_get_info(i)[0] / (1024**3)
    total_mem = torch.cuda.mem_get_info(i)[1] / (1024**3)
    print(f"  - GPU {i}: {torch.cuda.get_device_name(i)} | VRAM: {free_mem:.2f}/{total_mem:.2f} GiB khả dụng")

if gpu_count > 1:
    print("-> Tự động kích hoạt cơ chế phân rã mô hình đa GPU (device_map='auto') nhằm giải tỏa áp lực VRAM.")
else:
    print("-> Huấn luyện trên đơn GPU kết hợp cơ chế bộ nhớ phân trang Paged AdamW 8-bit và co gọn target_modules.")


In [ ]:
# ==============================================================================
# 6. HUẤN LUYỆN TINH CHỈNH THAM SỐ HIỆU QUẢ (QLORA SFT TRAINING)
# ==============================================================================

import os, subprocess, yaml
from pathlib import Path

adapter_dir = "checkpoints/task3/mixed_balanced"

if MODE == "qwen_qlora":
    print("=" * 70)
    print(">>> KHỞI CHẠY TIẾN TRÌNH HUẤN LUYỆN QWEN2.5-CODER-7B (QLORA 4-BIT)")
    print(f"    - Phân phối thực nghiệm: {EXPERIMENT} (Kết hợp Real-train & Synthetic-train)")
    print(f"    - Số lượng epochs: {EPOCHS}")
    print(f"    - Thư mục lưu trữ checkpoint: {adapter_dir}")
    print("=" * 70)

    # Hiệu chuẩn tham số tối ưu hóa VRAM và bỏ qua kiểm tra tiến trình cha Jupyter Kernel:
    cfg_path = "configs/task3/qlora.yaml"
    with open(cfg_path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f) or {}
    cfg["min_free_vram_gib"] = 10.0
    cfg["attn_implementation"] = "sdpa"
    cfg["target_modules"] = ["q_proj", "k_proj", "v_proj", "o_proj"]
    cfg["optim"] = "paged_adamw_8bit"
    cfg["ignore_competing_gpu_workload"] = True
    with open(cfg_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f)

    # Đồng bộ mã nguồn nội bộ src/task3/ để giải phóng kiểm tra tiến trình GPU & tích hợp đa GPU
    train_py = Path("src/task3/train.py")
    if train_py.exists():
        code = train_py.read_text(encoding="utf-8")
        if '"optim": config.get("optim"' not in code:
            code = code.replace('"fp16": not bf16,', '"fp16": not bf16,\n        "optim": config.get("optim", "paged_adamw_8bit"),')
        if 'device_map = config.get("device_map")' not in code:
            code = code.replace(
                '"device_map": {"": torch.cuda.current_device()},',
                'device_map = config.get("device_map") or ("auto" if torch.cuda.device_count() > 1 else {"": torch.cuda.current_device()})\n        model_kwargs["device_map"] = device_map'
            )
        if 'not config.get("ignore_competing_gpu_workload"' not in code:
            code = code.replace(
                'if workload["external_process_count"]:',
                'if not config.get("ignore_competing_gpu_workload", False) and workload["external_process_count"]:'
            )
        train_py.write_text(code, encoding="utf-8")

    preflight_py = Path("src/task3/gpu_preflight.py")
    if preflight_py.exists():
        code = preflight_py.read_text(encoding="utf-8")
        code = code.replace('if pid != os.getpid():', 'if pid not in {os.getpid(), os.getppid()}:')
        preflight_py.write_text(code, encoding="utf-8")

    train_cmd = [
        "python", "scripts/train_task3.py",
        "--config", "configs/task3/qlora.yaml",
        "--experiment", EXPERIMENT,
        "--output-dir", adapter_dir
    ]
    if MAX_STEPS:
        train_cmd.extend(["--max-steps", str(MAX_STEPS)])

    subprocess.run(train_cmd, check=True)
    print("\nTiến trình huấn luyện hoàn tất thành công. Trọng số LoRA adapter lưu tại:", adapter_dir)
else:
    print(f"Bỏ qua bước huấn luyện theo cấu hình thực thi hiện tại (MODE = '{MODE}').")


In [ ]:
# ==============================================================================
# 7. SUY LUẬN THEO LÔ VÀ KIỂM SOÁT MỨC ĐỘ SƯ PHẠM (BATCH INFERENCE)
# ==============================================================================

import os, json, subprocess, shutil
from pathlib import Path

output_file = f"outputs/task3/{MODE}_{EVAL_SPLIT}.predictions.json"
os.makedirs("outputs/task3", exist_ok=True)

# Khởi tạo ma trận lỗi phân loại định hướng (Taxonomy conditioning)
taxonomy_file = "outputs/task3/public_baseline_taxonomy.json"
if EVAL_SPLIT == "public":
    from scripts.build_task3_submission import build_default_taxonomy
    if not os.path.exists(taxonomy_file):
        build_default_taxonomy(Path("data/raw/public_test_release"), Path(taxonomy_file))

print(f"\nKhởi chạy tiến trình suy luận trên phân vùng dữ liệu: {EVAL_SPLIT}...")

infer_cmd = ["python", "scripts/run_task3.py", "--split", EVAL_SPLIT, "--output", output_file]

if EVAL_SPLIT == "public":
    infer_cmd.extend(["--taxonomy-source", "task2-json", "--taxonomy-json", taxonomy_file, "--allow-unverified-task2-export"])
else:
    infer_cmd.extend(["--taxonomy-source", "gold"])

if MODE == "gemini":
    infer_cmd.extend(["--config", "configs/task3/gemini.yaml"])
elif MODE == "openai":
    infer_cmd.extend(["--config", "configs/task3/openai.yaml"])
elif MODE == "qwen_baseline":
    infer_cmd.extend(["--config", "configs/task3/baseline.yaml"])
elif MODE == "qwen_qlora":
    target_adapter = f"{adapter_dir}/adapter" if os.path.exists(f"{adapter_dir}/adapter") else adapter_dir
    infer_cmd.extend(["--config", "configs/task3/baseline.yaml", "--adapter", target_adapter])

print("Lệnh thực thi:", " ".join(infer_cmd))
subprocess.run(infer_cmd, check=True)
print(f"\nĐã xuất bản kết quả suy luận tại: {output_file}")


In [ ]:
# ==============================================================================
# 8. ĐÁNH GIÁ ĐỘ TUÂN THỦ (COMPLIANCE AUDIT) VÀ ĐÓNG GÓI BẢN NỘP BÀI
# ==============================================================================

import os, json, shutil, subprocess

traces_file = output_file.replace(".predictions.json", ".predictions.traces.json")
eval_file = output_file.replace(".predictions.json", ".evaluation.json")

# 1. Đo lường các chỉ số tuân thủ sư phạm và kiểm định vi phạm rò rỉ lời giải
eval_cmd = [
    "python", "scripts/evaluate_task3.py",
    "--data-dir", "data/processed/task3",
    "--split", EVAL_SPLIT,
    "--predictions", output_file,
    "--traces", traces_file,
    "--taxonomy-source", ("task2-json" if EVAL_SPLIT == "public" else "gold"),
    "--output", eval_file
]
subprocess.run(eval_cmd, check=True)

# 2. Đóng gói tệp nộp bài chuẩn hóa ra thư mục làm việc chính
submission_dest = "/kaggle/working/task3_submission.json"
shutil.copy(output_file, submission_dest)
if os.path.exists(eval_file):
    shutil.copy(eval_file, f"/kaggle/working/{os.path.basename(eval_file)}")

print("\n" + "=" * 25 + " BÁO CÁO KIỂM ĐỊNH MỨC ĐỘ TUÂN THỦ (COMPLIANCE AUDIT) " + "=" * 25)
with open(eval_file, "r", encoding="utf-8") as f:
    eval_res = json.load(f)
    print(json.dumps(eval_res.get("summary", eval_res), indent=2, ensure_ascii=False))

# 3. Xác thực cấu trúc dữ liệu tệp nộp bài trước khi phát hành
with open(submission_dest, "r", encoding="utf-8") as f:
    preds = json.load(f)

print("\n" + "=" * 25 + " XÁC THỰC TÍNH TOÀN VẸN CỦA TỆP NỘP BÀI (SUBMISSION) " + "=" * 25)
print(f"Tổng số bản ghi dự đoán: {len(preds)} mẫu.")
print("Mẫu dữ liệu tham chiếu (3 mục đầu):")
print(json.dumps(preds[:3], indent=2, ensure_ascii=False))

print("\nDanh mục tệp sẵn sàng xuất bản trong /kaggle/working/:")
!ls -lh /kaggle/working
